# E-Commerce Sales Dashboard
Interactive Dash dashboard built on the preprocessed Brazilian e-commerce data.

In [1]:
import pandas as pd
import plotly.express as px
import dash
from dash import html, dcc
from dash.dependencies import Input, Output


In [2]:
data = pd.read_csv(r'C:\Users\ADMIN\Downloads\المبادرة\data\preprocessed.csv')

data['order_purchase_timestamp'] = pd.to_datetime(data['order_purchase_timestamp'])
data['order_delivered_customer_date'] = pd.to_datetime(data['order_delivered_customer_date'])

data = data[~data['order_status'].isin(['canceled', 'unavailable'])]
data = data.dropna(subset=['price', 'order_purchase_timestamp'])

data['year'] = data['order_purchase_timestamp'].dt.year
data['month'] = data['order_purchase_timestamp'].dt.month
data['weekday_num'] = data['order_purchase_timestamp'].dt.dayofweek
data['weekday'] = data['order_purchase_timestamp'].dt.day_name()
data['delivery_days'] = (data['order_delivered_customer_date']
                         - data['order_purchase_timestamp']).dt.days


In [3]:
app = dash.Dash()


In [4]:
app.layout = html.Div(children=[
    html.H1('E-Commerce Sales Dashboard',
            style={'textAlign': 'center', 'color': '#503D36', 'font-size': 40}),

    html.Div(["Year: ", dcc.Input(id='input-yr', value='2017', type='number',
                                  style={'height': '50px', 'font-size': 35}), ],
             style={'font-size': 40}),

    html.Div(["Category (optional): ", dcc.Input(id='input-cat', value='', type='text',
                                                 placeholder='e.g. bed_bath_table',
                                                 style={'height': '50px', 'font-size': 35}), ],
             style={'font-size': 40}),

    html.Br(),
    html.Br(),

    html.Div(dcc.Graph(id='monthly-plot')),
    html.Div(dcc.Graph(id='category-plot')),
    html.Div(dcc.Graph(id='weekday-plot')),
    html.Div(dcc.Graph(id='delivery-plot')),
])


In [5]:
def select_data(entered_year, entered_cat):

    if entered_year is None:
        entered_year = 2017
    df = data[data['year'] == int(entered_year)]
    if entered_cat and entered_cat.strip():
        df = df[df['product_category'] == entered_cat.strip().lower()]
    return df


In [6]:
@app.callback(Output(component_id='monthly-plot', component_property='figure'),
              [Input(component_id='input-yr', component_property='value'),
               Input(component_id='input-cat', component_property='value')])
def get_monthly(entered_year, entered_cat):
    df = select_data(entered_year, entered_cat)
    g1 = df.groupby('month')['price'].sum().reset_index()
    fig1 = px.line(g1, x='month', y='price',
                   title='Monthly Sales in year ' + str(entered_year or 2017),
                   labels={'month': 'Month', 'price': 'Sales'})
    fig1.update_layout()
    return fig1


In [7]:
@app.callback(Output(component_id='category-plot', component_property='figure'),
              [Input(component_id='input-yr', component_property='value'),
               Input(component_id='input-cat', component_property='value')])
def get_category(entered_year, entered_cat):
    df = select_data(entered_year, entered_cat)
    df = df[df['product_category'] != 'unknown']
    g2 = df.groupby('product_category')['price'].sum().nlargest(10).reset_index()
    fig2 = px.bar(g2, x='product_category', y='price',
                  title='Top 10 Categories by Sales in year ' + str(entered_year or 2017),
                  labels={'product_category': 'Category', 'price': 'Sales'})
    fig2.update_layout()
    return fig2


In [8]:
@app.callback(Output(component_id='weekday-plot', component_property='figure'),
              [Input(component_id='input-yr', component_property='value'),
               Input(component_id='input-cat', component_property='value')])
def get_weekday(entered_year, entered_cat):
    df = select_data(entered_year, entered_cat)
    g3 = df.groupby(['weekday_num', 'weekday'])['price'].sum().reset_index()
    fig3 = px.bar(g3, x='weekday', y='price',
                  title='Sales by Day of Week in year ' + str(entered_year or 2017),
                  labels={'weekday': 'Day', 'price': 'Sales'})
    fig3.update_layout()
    return fig3


In [9]:
@app.callback(Output(component_id='delivery-plot', component_property='figure'),
              [Input(component_id='input-yr', component_property='value'),
               Input(component_id='input-cat', component_property='value')])
def get_delivery(entered_year, entered_cat):
    df = select_data(entered_year, entered_cat)
    df = df.dropna(subset=['review_score', 'delivery_days'])
    df = df[df['delivery_days'] >= 0]
    g4 = df.groupby('review_score')['delivery_days'].mean().reset_index()
    fig4 = px.bar(g4, x='review_score', y='delivery_days',
                  title='Average Delivery Days by Review Score in year ' + str(entered_year or 2017),
                  labels={'review_score': 'Review Score',
                          'delivery_days': 'Average Delivery Days'})
    fig4.update_layout()
    return fig4


In [10]:
if __name__ == '__main__':
   app.run(port=8070, host='127.0.0.1', debug=True)
